# Séance 6 — Visualiser et analyser

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-medecine/notebooks/pym-s6-exercices.ipynb)

**Objectifs.** Construire des figures Matplotlib (figure, axes, subplots,
légende, seuil), tracer des séries temporelles avec un vrai axe de dates,
mener une analyse exploratoire avec seaborn, puis tester une différence
entre deux groupes et l'interpréter avec prudence.

**Données.**
- `constantes_vitales.csv` : 12 patients de réanimation, une mesure par heure pendant 48 h (données simulées) ;
- `covid_hospitalisations.csv.gz` : patients COVID-19 hospitalisés par département et par jour (Santé publique France, Licence Ouverte) ;
- `insuffisance_cardiaque.csv` : 299 patients en insuffisance cardiaque (Chicco & Jurman 2020, UCI, CC BY 4.0).

**Méthode.** Exécutez les cellules dans l'ordre. Complétez les cellules
marquées `# À vous`, puis exécutez la cellule de vérification qui suit :
elle affiche ✅ quand votre réponse est juste. Les questions *Lire du
code* et *Lire un graphique* se résolvent **avant** d'exécuter quoi que ce soit.

La correction est dans [`pym-s6-exercices-correction.ipynb`](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-medecine/notebooks/pym-s6-exercices-correction.ipynb) — à ouvrir après avoir cherché.

Exemples pédagogiques : aucun résultat de ce notebook n'est un outil de décision clinique.

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

URL = "https://raw.githubusercontent.com/racousin/data_science_practice/main/website/public/modules/python-medecine/donnees/"
sns.set_theme(style="whitegrid")

vitaux = pd.read_csv(URL + "constantes_vitales.csv")
covid = pd.read_csv(URL + "covid_hospitalisations.csv.gz",
                    dtype={"dep": str}, parse_dates=["jour"])
ic = pd.read_csv(URL + "insuffisance_cardiaque.csv")
print(vitaux.shape, covid.shape, ic.shape)

In [ ]:
vitaux.head()

---
## Partie A — Constantes vitales en réanimation (Matplotlib)

`vitaux` est au format **long** : une ligne par patient et par heure, avec
la fréquence cardiaque (bpm), la pression artérielle systolique (mmHg),
la SpO₂ (%) et la température (°C).

### Exercice 1 — Une courbe

Tracez la fréquence cardiaque du patient `R01` en fonction de l'heure,
avec l'interface objet (`fig, ax = plt.subplots()`). Nommez l'axe x
`"Heure"`, l'axe y avec son unité (`"Fréquence cardiaque (bpm)"`), et
donnez un titre.

In [ ]:
# À vous
p01 = ...          # les lignes du patient R01
fig, ax = plt.subplots(figsize=(9, 3.5))
# tracez, puis ax.set_xlabel(...), ax.set_ylabel(...), ax.set_title(...)
...

In [ ]:
assert isinstance(p01, pd.DataFrame) and len(p01) == 48, "p01 doit contenir les 48 lignes de R01"
assert len(ax.get_lines()) == 1, "il faut exactement une courbe"
assert len(ax.get_lines()[0].get_xdata()) == 48
assert ax.get_xlabel() == "Heure"
assert "bpm" in ax.get_ylabel(), "l'unité manque sur l'axe y"
assert ax.get_title() != "", "ajoutez un titre"
print("✅ Exercice 1 réussi")

### Lire du code — une grille d'axes

Que va afficher ce code ? Répondez **avant** d'exécuter.

```python
fig, axes = plt.subplots(2, 2)
print(type(axes).__name__, axes.shape)
print(axes[1, 0] is axes.flat[2])
axes[0, 1].set_title("SpO2")
print([a.get_title() for a in axes.flat])
```

*Votre réponse :* …

In [ ]:
fig, axes = plt.subplots(2, 2)
print(type(axes).__name__, axes.shape)
print(axes[1, 0] is axes.flat[2])
axes[0, 1].set_title("SpO2")
print([a.get_title() for a in axes.flat])
plt.close(fig)

### Exercice 2 — Les quatre constantes d'un patient

Écrivez `tracer_constantes(df, patient)` qui crée une grille 2 × 2
(`sharex=True`) et trace, pour ce patient, une constante par case :
fréquence cardiaque, pression systolique, SpO₂, température. Chaque case
a un titre (le nom de la constante) ; la figure a un titre général qui
contient l'identifiant du patient (`fig.suptitle`). La fonction renvoie
`fig, axes`. Appliquez-la au patient `R03`.

In [ ]:
CONSTANTES = {
    "frequence_cardiaque": "Fréquence cardiaque (bpm)",
    "pression_systolique": "Pression systolique (mmHg)",
    "spo2": "SpO2 (%)",
    "temperature": "Température (°C)",
}

def tracer_constantes(df, patient):
    # À vous : filtrer le patient, plt.subplots(2, 2, ...), une boucle sur
    # zip(axes.flat, CONSTANTES.items()), fig.suptitle(...), fig.tight_layout()
    ...

fig, axes = tracer_constantes(vitaux, "R03")

In [ ]:
assert fig is not None and axes is not None, "la fonction doit renvoyer fig, axes"
assert axes.shape == (2, 2)
assert all(len(a.get_lines()) == 1 for a in axes.flat), "une courbe par case"
titres = [a.get_title() for a in axes.flat]
assert all(titres) and len(set(titres)) == 4, "quatre titres différents"
assert fig._suptitle is not None and "R03" in fig._suptitle.get_text(), "le titre général doit nommer le patient"
spo2 = axes.flat[2].get_lines()[0].get_ydata()
assert min(spo2) == vitaux.loc[vitaux["id_patient"] == "R03", "spo2"].min()
print("✅ Exercice 2 réussi")

Que voyez-vous chez R03 après la 20ᵉ heure ? Quelle association de signes
cela évoque-t-il ?

*Votre réponse :* …

### Exercice 3 — Douze patients, trois qui se dégradent

1. Calculez `deteriores` : la liste triée des patients dont la SpO₂ est
   descendue **sous 92 %** au moins une fois.
2. Tracez la SpO₂ des 12 patients sur un même axe : les patients non
   détériorés en gris fin (`color="lightgrey"`, `linewidth=1`), les
   patients de `deteriores` en couleur avec un `label`, puis une ligne
   horizontale pointillée à 92 % (`ax.axhline`) et une légende.

Le seuil de 92 % est un repère pédagogique (borne basse de la cible
usuelle de SpO₂ chez l'adulte sans insuffisance respiratoire chronique).

In [ ]:
SEUIL = 92
deteriores = ...   # À vous : groupby + min, puis filtrer
fig, ax = plt.subplots(figsize=(10, 4))
# À vous : une boucle sur vitaux.groupby("id_patient"), puis ax.axhline(...)
...

In [ ]:
assert deteriores == ["R03", "R07", "R11"], f"deteriores = {deteriores}"
lignes = ax.get_lines()
assert len(lignes) == 13, "12 courbes + la ligne du seuil"
assert any(list(l.get_ydata()) == [SEUIL, SEUIL] for l in lignes), "ligne du seuil absente"
assert ax.get_legend() is not None, "ajoutez une légende"
noms = [t.get_text() for t in ax.get_legend().get_texts()]
assert all(p in noms for p in deteriores), "la légende doit nommer les 3 patients"
print("✅ Exercice 3 réussi")

### Exercice 4 — Trouvez l'erreur

Ce code doit tracer, côte à côte, la fréquence cardiaque et la
température du patient R07. Il plante. Lisez le message d'erreur,
expliquez-le, puis corrigez le code (en gardant `plt.subplots(1, 2)`).

In [ ]:
# À vous : corrigez
p07 = vitaux[vitaux["id_patient"] == "R07"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax.plot(p07["heure"], p07["frequence_cardiaque"])
ax.plot(p07["heure"], p07["temperature"])
ax.set_xlabel("Heure")

In [ ]:
assert len(ax) == 2 and all(len(a.get_lines()) == 1 for a in ax), "une courbe par case"
assert all(a.get_xlabel() == "Heure" for a in ax), "nommez l'axe x des deux cases"
print("✅ Exercice 4 réussi")

---
## Partie B — Hospitalisations COVID-19 en France

Chaque ligne de `covid` : un département (`dep`) et un jour (`jour`).
`hosp` et `rea` comptent les patients **présents** ce jour-là à l'hôpital
et en réanimation (un stock) ; `rad` et `dc` sont des **cumuls** depuis
mars 2020 (retours à domicile, décès à l'hôpital).

In [ ]:
covid.head()

### Exercice 5 — La courbe nationale

1. Construisez `national` : la somme de `hosp` et `rea` sur tous les
   départements, pour chaque jour (index = `jour`).
2. Tracez les deux séries sur un même axe (même unité : des patients),
   avec un vrai axe de dates : `mdates.ConciseDateFormatter` sur
   `ax.xaxis`. Légende, titre, étiquette y.
3. Calculez `pic_hosp`, la date du maximum de `hosp`.

In [ ]:
national = ...      # À vous : groupby("jour") puis sum()
pic_hosp = ...
fig, ax = plt.subplots(figsize=(11, 4))
# À vous : deux ax.plot avec label, puis l'axe des dates :
# loc = mdates.AutoDateLocator(); ax.xaxis.set_major_locator(loc)
# ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
...

In [ ]:
assert isinstance(national.index, pd.DatetimeIndex), "jour doit être une date (parse_dates)"
assert len(national) == covid["jour"].nunique()
assert national["hosp"].max() == 33466 and str(pic_hosp.date()) == "2020-11-16"
assert national["rea"].max() == 7019
assert len(ax.get_lines()) == 2 and ax.get_legend() is not None
assert isinstance(ax.xaxis.get_major_formatter(), mdates.ConciseDateFormatter)
print("✅ Exercice 5 réussi")

Combien de vagues voyez-vous ? Le pic des réanimations a-t-il lieu à la
même vague que le pic des hospitalisations ?

*Votre réponse :* …

### Exercice 6 — Trois départements

Comparez les hospitalisations de Paris (`75`), des Bouches-du-Rhône
(`13`) et du Rhône (`69`). Construisez `trois` au format **large**
(une colonne par département, index = jour) avec `pivot`, puis
tracez-le (une courbe par colonne, légende).

In [ ]:
DEPS = ["75", "13", "69"]
trois = ...      # À vous : filtrer avec isin, puis pivot(index=..., columns=..., values=...)
fig, ax = plt.subplots(figsize=(11, 4))
...

In [ ]:
assert sorted(trois.columns) == ["13", "69", "75"]
assert trois["75"].max() == 3281 and str(trois["75"].idxmax().date()) == "2020-04-14"
assert len(ax.get_lines()) == 3 and ax.get_legend() is not None
print("✅ Exercice 6 réussi")

Le pic parisien est au printemps 2020, celui du Rhône et des
Bouches-du-Rhône à l'automne 2020. Peut-on comparer directement les
hauteurs des courbes pour dire quel département a été le plus touché ?

*Votre réponse :* …

### Exercice 7 — Décès quotidiens et moyenne glissante sur 7 jours

`dc` est un cumul. Le nombre de décès d'un jour est la différence avec la
veille (`.diff()`). Cette série quotidienne est bruitée (moins de
déclarations le week-end) : on la lisse par une **moyenne glissante sur
7 jours** (`.rolling(7).mean()`).

Construisez la série `deces_jour` (somme nationale de `dc`, puis
`diff`) et `deces_7j`, puis tracez les deux sur le même axe : la série
brute en gris clair, la moyenne glissante par-dessus.

In [ ]:
deces_jour = ...   # À vous
deces_7j = ...
fig, ax = plt.subplots(figsize=(11, 4))
...

In [ ]:
assert deces_jour.isna().sum() == 1, "diff() laisse un seul NaN, le premier jour"
assert deces_jour.max() == 605
assert round(deces_7j.max(), 1) == 514.3
assert str(deces_7j.idxmax().date()) == "2020-04-08"
assert deces_7j.isna().sum() == 7, "rolling(7) : les 6 premiers jours + le NaN de diff"
assert len(ax.get_lines()) == 2
print("✅ Exercice 7 réussi")

### Lire un graphique — deux axes y

Exécutez la cellule, puis répondez : que semble montrer cette figure ?
En quoi est-elle trompeuse ?

In [ ]:
nat = covid.groupby("jour")[["hosp", "dc"]].sum()
d7 = nat["dc"].diff().rolling(7).mean()
fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(nat.index, nat["hosp"], color="tab:blue")
ax1.set_ylabel("Hospitalisés", color="tab:blue")
ax2 = ax1.twinx()                      # un second axe y, à droite
ax2.plot(d7.index, d7, color="tab:red")
ax2.set_ylabel("Décès par jour (7 j)", color="tab:red")
ax2.set_ylim(0, 1200)
ax1.set_title("Les décès ne suivent plus les hospitalisations ?");

*Votre réponse :* …

---
## Partie C — Analyse exploratoire : insuffisance cardiaque

299 patients suivis après une hospitalisation pour insuffisance
cardiaque. `deces` vaut 1 si le patient est décédé pendant le suivi.
Unités : `fraction_ejection` en %, `creatinine` en mg/dL, `sodium` en
mmol/L, `cpk` en UI/L, `plaquettes` en 10³/µL.

**Question** : quelles mesures relevées à l'inclusion sont associées au décès ?

In [ ]:
ic.describe().T.round(2)

### Exercice 8 — Distributions

1. Sur une grille 1 × 3, tracez l'histogramme (`sns.histplot`, `ax=`)
   de `age`, `fraction_ejection` et `creatinine`.
2. Calculez `moy_creat` et `med_creat`, la moyenne et la médiane de la
   créatinine. Laquelle des trois distributions est la plus asymétrique ?
3. Sur une nouvelle figure `fig2, ax2`, tracez la boîte à moustaches de
   `fraction_ejection` selon `deces` (`sns.boxplot`), et calculez
   `medianes_fe`, la médiane de la fraction d'éjection par groupe.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
# À vous : trois sns.histplot(data=ic, x=..., ax=...)
...
moy_creat = ...
med_creat = ...

fig2, ax2 = plt.subplots(figsize=(5, 4))
# À vous : sns.boxplot(...)
medianes_fe = ...

In [ ]:
assert axes.shape == (3,) and all(len(a.patches) > 0 for a in axes), "trois histogrammes"
assert [a.get_xlabel() for a in axes] == ["age", "fraction_ejection", "creatinine"]
assert round(moy_creat, 2) == 1.39 and med_creat == 1.1
assert ax2.get_ylabel() != "" and len(ax2.patches) + len(ax2.lines) > 0
assert list(medianes_fe) == [38.0, 30.0]
print("✅ Exercice 8 réussi")

*Votre réponse :* …

### Lire un graphique — un axe tronqué

Que semble montrer cette figure ? En quoi est-elle trompeuse ?

In [ ]:
taux = ic.groupby("anemie")["deces"].mean() * 100
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.bar(["Sans anémie", "Avec anémie"], taux, color=["tab:blue", "tab:red"])
ax.set_ylim(28, 37)
ax.set_ylabel("Décès pendant le suivi (%)")
ax.set_title("L'anémie, facteur de mortalité majeur");

*Votre réponse :* …

### Exercice 9 — Relations entre variables

1. Nuage de points `age` × `creatinine` coloré par `deces`
   (`sns.scatterplot`, `hue=`), sur `fig, ax`.
2. `corr` : la matrice des corrélations de **Spearman** de toutes les
   colonnes numériques (sans `id_patient`), affichée avec `sns.heatmap`
   (`annot=True`, `fmt=".2f"`, `cmap="vlag"`, `center=0`) sur `fig3, ax3`.
3. `var_max` : le nom de la variable dont la corrélation avec `deces`
   est la plus forte **en valeur absolue** (hors `deces` lui-même).
4. Un `sns.pairplot` de `age`, `fraction_ejection`, `creatinine`,
   `sodium`, coloré par `deces` (stockez le résultat dans `g`).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
...    # À vous : le nuage de points
corr = ...
fig3, ax3 = plt.subplots(figsize=(10, 8))
...    # À vous : la heatmap
var_max = ...
g = ...

In [ ]:
assert len(ax.collections) > 0 and ax.get_legend() is not None, "nuage coloré par deces"
assert corr.shape == (13, 13) and "id_patient" not in corr.columns
assert round(corr.loc["creatinine", "deces"], 2) == 0.37, "corrélation de Spearman ?"
assert len(ax3.collections) > 0, "heatmap absente"
assert var_max == "suivi_jours"
assert g.axes.shape == (4, 4)
print("✅ Exercice 9 réussi")

`suivi_jours` est la variable la plus corrélée au décès (−0,54). Est-ce
un facteur de risque ? Pourquoi la créatinine est-elle plus corrélée au
décès en Spearman (0,37) qu'en Pearson (0,29) ?

*Votre réponse :* …

### Exercice 10 — Tester, puis interpréter

1. `p_mw` : la p-valeur du test de **Mann-Whitney** comparant la
   fraction d'éjection des survivants et des patients décédés
   (`stats.mannwhitneyu`). Comparez avec le test de Welch
   (`stats.ttest_ind(..., equal_var=False)`), `p_t`.
2. `table` : le tableau de contingence `anemie` × `deces`
   (`pd.crosstab`) ; `p_chi2` : la p-valeur du test du χ²
   (`stats.chi2_contingency`, qui renvoie un objet dont l'attribut
   `pvalue` est la p-valeur).
3. Rédigez l'interprétation des deux résultats en trois ou quatre
   phrases, dans la cellule de texte qui suit.

In [ ]:
fe_survivants = ...
fe_deces = ...
p_mw = ...
p_t = ...
table = ...
p_chi2 = ...

In [ ]:
assert 1e-7 < p_mw < 1e-6, f"p_mw = {p_mw}"
assert 1e-6 < p_t < 1e-4, f"p_t = {p_t}"
assert table.shape == (2, 2) and table.values.sum() == 299
assert round(p_chi2, 2) == 0.31, f"p_chi2 = {p_chi2}"
print("✅ Exercice 10 réussi")

*Votre réponse :* …

---
### Pour aller plus loin — tester six variables à la fois

Construisez `resume`, un DataFrame indexé par les variables `age`,
`cpk`, `fraction_ejection`, `plaquettes`, `creatinine`, `sodium`, avec
les colonnes `mediane_survivants`, `mediane_deces`, `p_valeur`
(Mann-Whitney) et `p_bonferroni` (= `p_valeur × 6`, plafonnée à 1),
trié par `p_valeur` croissante. Tracez ensuite une figure 2 × 3 avec un
`sns.boxplot` par variable (`x="deces"`), la p-valeur dans le titre de
chaque case. Pourquoi corriger les p-valeurs quand on fait plusieurs tests ?

In [ ]:
VARIABLES = ["age", "cpk", "fraction_ejection", "plaquettes", "creatinine", "sodium"]
resume = ...   # À vous

In [ ]:
assert list(resume.index[:2]) == ["creatinine", "fraction_ejection"]
assert (resume["p_bonferroni"] < 0.05).sum() == 4
assert resume.loc["cpk", "p_bonferroni"] == 1
print("✅ Pour aller plus loin réussi")

*Votre réponse :* …